In [ ]:
import torch
import os
from torch.utils.data import Dataset, DataLoader, random_split, Subset, WeightedRandomSampler
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import random
import pickle
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import re
from MambaFRZ import initialize_mamba2_predictor
from SmartFRZ import initialize_smartfrz_predictor
from CompressedFreezeDataset_ImportanceSampling import CompressedFreezeDataset
from collections import defaultdict
from tqdm import tqdm
from torchinfo import summary
from data_generation import generate_formatted_data, generate_compressed_dataset
        
def main(args):
  root_dir = f"{args.name_of_experiment}/context_window_{args.context_window_size}"
  total_count = args.number_of_samples
  counter = 0
  name_of_experiment = args.name_of_experiment
  window_size = args.context_window_size
  if args.generate_training_data:
      generate_compressed_dataset(root_dir, total_count, args.frz_predictor_type)
  
  path_for_importance_sample_weights = f"{root_dir}/sample_importance_weights_{args.importance_sampling_type}_{args.frz_predictor_type}.pkl"
  with open(path_for_importance_sample_weights, "rb") as f:
      sample_importance_weights = pickle.load(f)
  train_dataset = CompressedFreezeDataset(f"{root_dir}/compressed_dataset_{args.frz_predictor_type}.pkl", args.frz_predictor_type, sample_importance_weights)
  all_indices = list(range(len(train_dataset)))
    
  seed_to_indices = defaultdict(list)
  for idx in range(len(train_dataset)):
    _, _, _, seed, _ = train_dataset[idx][0]
    seed_to_indices[seed].append(idx)

  print("The seeds in the dataset: ", seed_to_indices.keys())
  print("The number of entries per seed: ", [len(seed_indices) for seed_indices in seed_to_indices.values()])

  # REPRODUCIBILITY WITH RANDOM SEED
  random.seed(1234)
  chosen_seed = random.choice(list(seed_to_indices.keys())) # or e.g. '42'
  torch.manual_seed(1234)
  print(f"The chosen seed: {chosen_seed}")
  val_indices = seed_to_indices[chosen_seed]

  train_indices = [idx for seed, indices in seed_to_indices.items() if seed != chosen_seed for idx in indices]
    
  train_subset = Subset(train_dataset, train_indices)
  val_subset = Subset(train_dataset, val_indices)
    
  batch_size = 32
  num_workers = 0
  train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
  val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
  print(f"Number of batches in Train Loader: {len(train_loader)}")
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  if args.frz_predictor_type == "smartfrz":
    re_size = args.re_size
    in_channel = re_size
    hid_channel = 256
    out_channel = 64
    predictor = initialize_smartfrz_predictor(in_channel, hid_channel, out_channel)
    predictor.load_state_dict(torch.load(args.pretrained_weights, map_location=device))
  elif args.frz_predictor_type == "mambafrz":
    feature_dim = args.re_size
    mlp_hid_channel = 512
    mlp_out_channel = 2
    ssm_state_expansion_factor = 32
    projected_dim = feature_dim // 2
    predictor = initialize_mamba2_predictor(feature_dim=feature_dim, projected_dim=projected_dim, ssm_state_expansion_factor=ssm_state_expansion_factor, mlp_hid_channel=mlp_hid_channel, mlp_out_channel=mlp_out_channel)
    predictor.load_state_dict(torch.load(args.pretrained_weights, map_location=device))
    
  # Simple parameter count
  total_params = sum(p.numel() for p in predictor.parameters())
  trainable_params = sum(p.numel() for p in predictor.parameters() if p.requires_grad)
  # print(summary(predictor, input_size=(1, args.context_window_size, args.re_size)))

  print(f"Total parameters: {total_params:,}")
  print(f"Trainable parameters: {trainable_params:,}")
  
  predictor.to(device)

  label_smoothing = 0.2 # included label smoothing

  criterion = nn.CrossEntropyLoss()
  criterion = criterion.to(device)

  if args.frz_predictor_type == "smartfrz":
    optimizer = optim.AdamW(predictor.parameters(),
                        lr=1e-4,    # try 1e-3 then 1e-4
                        weight_decay=1e-5)  # small L2 to regularize
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True)
  elif args.frz_predictor_type == "mambafrz":
    optimizer = optim.AdamW(predictor.parameters(),
                        lr=1e-5,    # try 1e-3 then 1e-4
                        weight_decay=1e-5)  # small L2 to regularize
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True)
  
  num_epochs = args.num_epochs
  
  model_save_path = f"{root_dir}/{args.frz_predictor_type}_{args.checkpoint_folder}"
  os.makedirs(model_save_path, exist_ok=True)
  
  frozen_count = 0
  non_frozen_count = 0
  best_training_acc = 0.0
  
  training_epoch_loss = []
  for epoch in range(num_epochs):
    predictor.train()
    train_running_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
    for inputs, labels in progress_bar:
      layer_names_list = inputs[1]
      epoch_list = inputs[2]
      seed_list = inputs[3]
      inputs, labels = inputs[0].to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = predictor(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      torch.nn.utils.clip_grad_norm_(predictor.parameters(), max_norm=1.0)
      optimizer.step()
      train_running_loss += loss.item()
      correct += sum([torch.argmax(pred).item() == label.item() for pred, label in zip(outputs, labels)])
      total += labels.size(0)
    train_running_loss /= len(train_loader)
    training_epoch_loss.append(train_running_loss)
    epoch_accuracy = correct / total
    print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {train_running_loss:.4f}, Training Accuracy: {epoch_accuracy:.4f}")
    
    # Begin validation
    predictor.eval()
    val_correct_by_layer = {}
    layer_by_layer_predictions = defaultdict(list)
    val_total_by_layer = {}
    val_total_correct = 0
    val_total_num = 0
    val_running_loss = 0.0
    
    progress_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
    
    with torch.no_grad():
      for inputs, labels in progress_bar:
        layer_names_list = inputs[1]
        epoch_numbers_list = inputs[2]
        sample_weights_list = inputs[4]
        inputs, labels = inputs[0].to(device), labels.to(device)
        outputs = predictor(inputs)
        loss = criterion(outputs, labels)
        val_running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
            
        for pred, label, layer_name, epoch_num, sample_weight in zip(preds, labels, layer_names_list, epoch_numbers_list, sample_weights_list):
          if layer_name not in val_correct_by_layer:
            val_correct_by_layer[layer_name] = 0
            val_total_by_layer[layer_name] = 0
          if pred.item() == label.item():
            val_correct_by_layer[layer_name] += 1 * (1.0 + sample_weight)
            val_total_correct += 1 * (1.0 + sample_weight)
          val_total_by_layer[layer_name] += 1 * (1.0 + sample_weight)
          layer_by_layer_predictions[layer_name].append((epoch_num, pred.item(), label.item()))
          val_total_num += 1 * (1.0 + sample_weight)
    for layer_name, frz_predictions_by_layer in layer_by_layer_predictions.items():
        frz_predictions_by_layer.sort(key=lambda item: int(item[0]))
        epoch_list = [int(item[0]) for item in frz_predictions_by_layer]
        frz_predictor_list = [item[1] for item in frz_predictions_by_layer]
        label_predictor_list = [item[2] for item in frz_predictions_by_layer]
        plt.title(f"Layer: {layer_name}, Epoch {epoch} Predictions")
        plt.plot(epoch_list, frz_predictor_list, label="MambaFRZ Predictions")
        plt.plot(epoch_list, label_predictor_list, label="Labels")
        plt.legend()
        plt.show()
        
    val_running_loss /= len(val_loader)
    scheduler.step(val_running_loss)
    
    print(f"Validation Accuracy by Layer for Epoch {epoch + 1}:")
    for layer in sorted(val_correct_by_layer.keys()):
        acc = val_correct_by_layer[layer] / val_total_by_layer[layer]
        print(f"  {layer}: {acc:.4f}, {val_correct_by_layer[layer]} / {val_total_by_layer[layer]}")
    print(f"Overall Validation Accuracy: {(val_total_correct / val_total_num):.4f}")
    
    if epoch_accuracy > best_training_acc:
      best_training_acc = epoch_accuracy
      torch.save(predictor.state_dict(), os.path.join(model_save_path, f"{args.frz_predictor_type}_{epoch}.pth"))
      print(f"New Best Acc: {epoch_accuracy}")
  plt.plot(training_epoch_loss, label='Training Loss')
  plt.legend()
  plt.show()
    
    
      
class Args:
  def __init__(self, name_of_experiment, context_window_size, number_of_samples, re_size=1024, num_epochs=2, generate_training_data=False, checkpoint_folder="checkpoints", frz_predictor_type="smartfrz", importance_sampling_type="quad", pretrained_weights="test.pth"):
    self.context_window_size = context_window_size
    self.name_of_experiment = name_of_experiment
    self.number_of_samples = number_of_samples
    self.re_size = re_size
    self.num_epochs = num_epochs
    self.generate_training_data = generate_training_data
    self.checkpoint_folder = checkpoint_folder
    self.frz_predictor_type = frz_predictor_type
    self.importance_sampling_type = importance_sampling_type
    self.pretrained_weights = pretrained_weights

args = Args(name_of_experiment="mambafrz_vgg16_data_generation/training_data", context_window_size=30, number_of_samples=50000, re_size=1024, num_epochs=10, generate_training_data=False, checkpoint_folder="from_scratch_do_not_interfere", frz_predictor_type="smartfrz", importance_sampling_type="quad", pretrained_weights="mambafrz_vgg11_data_generation_12_seeds/training_data_more_data/context_window_30/smartfrz_small_test/smartfrz_9.pth")
main(args)